In [1]:
# Automatically reloads all code changes in python modules when changes are saved
%load_ext autoreload
%autoreload 2

In [2]:
## Initialize logger for whole project
## Also tracks output of YOLO ML training to this log file

from logging_config import setup_logging
from ultralytics import YOLO

# LOG_FILE = 'flower_classifier.log'
# logger = setup_logging(log_file=str(LOG_FILE))
logger = setup_logging()
logger.debug(f"Logging setup successful.")

2026-04-06 11:01:26,283 - flower_classifier - INFO - Writing log to: '/home/jdc6281/Coding/thesis/flower-classifier/process/flower_classifier.log'


In [13]:
## Sets up environment variables with kaggle
## MUST HAVE .env FILE WITH FILLED IN KAGGLE LOGIN INFO EVEN IF USE_KAGGLE IS FALSE
from dotenv import load_dotenv
from pathlib import Path
import os

file_path = ".env"
if not os.path.isfile(file_path):
    logger.error("'.env' file not found in this directory. Please make .env file with KAGGLE_USERNAME and KAGGLE_KEY. See README.md for more information.")
else:
    logger.info(".env file found! Importing configs.")

# Extract environment variables from .env file
load_dotenv()
KAGGLE_USERNAME = os.getenv('KAGGLE_USERNAME')
KAGGLE_KEY = os.getenv('KAGGLE_KEY')
GOOGLE_ZIP_ID = os.getenv('GOOGLE_ZIP_ID')

# Resolve project root directory
path = Path(__name__).resolve().parent
while path.name != "flower-classifier" and path != path.parent:
    path = path.parent
logger.info(f"Path to project parent directory: '{path}'")
PROJECT_ROOT = path

2026-04-06 11:47:24,287 - flower_classifier - INFO - .env file found! Importing configs.
2026-04-06 11:47:24,297 - flower_classifier - INFO - Path to project parent directory: '/home/jdc6281/Coding/thesis/flower-classifier'


In [4]:
#from get_dataset import get_dataset
#from ingest import ingest
#from config_IO import save_json_config, load_json_config, make_YOLO26_config
#from system_info import print_system_info, verify_venv
#from view_file import view_image, view_text

import get_dataset
import ingest
import config_IO
import system_info
import view_file

from pathlib import Path
import os
import sys
import platform
import json
import multiprocessing
import yaml
import torch

2026-04-06 11:01:32,544 - flower_classifier.get_dataset - INFO - Path to project parent directory: '/home/jdc6281/Coding/thesis/flower-classifier'


In [5]:
system_info.verify_venv()

2026-04-06 11:01:37,568 - flower_classifier.system_info - INFO - Jupyter is running in venv!


True

In [7]:
USE_KAGGLE = False

do_unzip = get_dataset.get_dataset(from_kaggle=USE_KAGGLE, kaggle_username=KAGGLE_USERNAME, kaggle_key=KAGGLE_KEY, google_zip_id=GOOGLE_ZIP_ID)

if do_unzip:
    get_dataset.unzip_dataset()

2026-04-06 11:04:57,228 - flower_classifier.get_dataset - INFO - Path to project parent directory: '/home/jdc6281/Coding/thesis/flower-classifier'
2026-04-06 11:04:58,902 - flower_classifier.get_dataset - INFO - Retrieving dataset from google drive...
2026-04-06 11:04:58,903 - flower_classifier.get_dataset - INFO - Downloading compressed dataset via Google Drive to '/home/jdc6281/Coding/thesis/flower-classifier/data/compressed/tomato-flower-3-class.zip'


Downloading...
From (original): https://drive.google.com/uc?id=1GWKMcoQ52PxBDwIWmCX1JEPnmucnyNl-
From (redirected): https://drive.google.com/uc?id=1GWKMcoQ52PxBDwIWmCX1JEPnmucnyNl-&confirm=t&uuid=f6f35b44-baad-419d-85c8-d752f92535bb
To: /home/jdc6281/Coding/thesis/flower-classifier/data/compressed/tomato-flower-3-class.zip
100%|████████████████████████████████████████| 711M/711M [00:18<00:00, 37.9MB/s]


2026-04-06 11:05:26,073 - flower_classifier.get_dataset - INFO - Download complete!
2026-04-06 11:05:26,074 - flower_classifier.get_dataset - INFO - Extracting files to directory: '/home/jdc6281/Coding/thesis/flower-classifier/data/data_set'
2026-04-06 11:16:08,354 - flower_classifier.get_dataset - INFO - Files extracted successfully!


In [8]:
ingest.ingest()

2026-04-06 11:16:23,481 - flower_classifier.ingest - INFO - Total source images: 3636
2026-04-06 11:16:23,483 - flower_classifier.ingest - INFO - Validation images:   1818
2026-04-06 11:16:23,483 - flower_classifier.ingest - INFO - Test images:         1818
2026-04-06 11:22:25,296 - flower_classifier.ingest - INFO - DONE! Split complete.
2026-04-06 11:22:28,212 - flower_classifier.ingest - INFO -              
 TRAIN:             
   images = 14545             
   labels = 14545             
 
2026-04-06 11:22:28,316 - flower_classifier.ingest - INFO -              
 VAL:             
   images = 1818             
   labels = 1818             
 
2026-04-06 11:22:28,416 - flower_classifier.ingest - INFO -              
 TEST:             
   images = 1818             
   labels = 1818             
 
2026-04-06 11:22:28,436 - flower_classifier.ingest - INFO - DONE! Wrote data.yaml to: /home/jdc6281/Coding/thesis/flower-classifier/config/data.yaml
2026-04-06 11:22:28,446 - flower_classifi

In [15]:
config_IO.make_YOLO26_config()

2026-04-06 12:16:12,122 - flower_classifier.config_IO - INFO - Path to project parent directory: '/home/jdc6281/Coding/thesis/flower-classifier'
2026-04-06 12:16:12,125 - flower_classifier.config_IO - INFO - Creating default config set at location '/home/jdc6281/Coding/thesis/flower-classifier/config/YOLO26_config.json'
2026-04-06 12:16:12,127 - flower_classifier.config_IO - INFO - Default config set: 
{
    "model_cfg": "yolo26n.yaml",
    "data_yaml": "/home/jdc6281/Coding/thesis/flower-classifier/config/data.yaml",
    "project_dir": "/home/jdc6281/Coding/thesis/flower-classifier/runs",
    "run_name": "/home/jdc6281/Coding/thesis/flower-classifier/tomato_flower_yolo26_from_scratch",
    "train_params": {
        "epochs": 100,
        "imgsz": 640,
        "batch": -1,
        "device": -1,
        "workers": 8,
        "patience": 30,
        "optimizer": "auto",
        "lr0": 0.01,
        "lrf": 0.01,
        "momentum": 0.937,
        "weight_decay": 0.0005,
        "warmup_ep

In [10]:
system_info.print_system_info()

2026-04-06 11:22:43,614 - flower_classifier.system_info - INFO -          
 torch 2.11.0+cu130         
 torchvision 0.26.0+cu130         
 ultralytics 8.4.33         
 cv2 4.13.0         
 captum 0.7.0         
 CUDA available: True         
 Python 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]         
 Executable /home/jdc6281/Coding/thesis/.venv/bin/python         
 Platform Linux-5.15.0-173-generic-x86_64-with-glibc2.35         



In [ ]:
## CPU Usage Config


CPU_THREADS = 4

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)

torch.set_num_threads(CPU_THREADS)

logger.info("Using CPU threads:", CPU_THREADS)

In [ ]:
## Checks ALL system configs for current settings.
## Also catches many common setup errors or issues.


# ==========================
# USER CONFIG
# ==========================
DATA_YAML =  PROJECT_ROOT / "config/data.yaml"
DATASET_ROOT = PROJECT_ROOT / "data/data_set/tomato-flower"
PROJECT_DIR = PROJECT_ROOT / "runs"
RUN_NAME = "tomato_flower_yolo26_from_scratch"
MODEL_CFG = "yolo26n.yaml"   # change later if needed

# ==========================
# HELPERS
# ==========================
def count_files(path, exts=None):
    if not path.exists():
        return 0
    files = [p for p in path.iterdir() if p.is_file()]
    if exts:
        exts = {e.lower() for e in exts}
        files = [p for p in files if p.suffix.lower() in exts]
    return len(files)

def print_path_status(label, path):
    path = Path(path)
    logger.info(f" \
        \n {label}: \
        \n  path   = {path} \
        \n  exists = {path.exists()} \
        \n  is_dir = {path.is_dir()} \
        \n  is_file= {path.is_file()} \
        \n")
    
# ==========================
# SYSTEM INFO
# ==========================
logger.info(f" \
    \n === SYSTEM INFO === \
    \n Python executable: {sys.executable} \
    \n Python version   : {sys.version.splitlines()[0]} \
    \n Platform         : {platform.platform()} \
    \n Machine          : {platform.machine()} \
    \n Processor        : {platform.processor()} \
    \n CPU count        : {multiprocessing.cpu_count()} \
    \n Working dir      : {Path.cwd() } \
    \n ")

logger.info(f" \
    \n === TORCH INFO === \
    \n torch version    : {torch.__version__} \
    \n CUDA available   : {torch.cuda.is_available()} \
    \n MPS available    : {hasattr(torch.backends, "mps") and torch.backends.mps.is_available()} \
    \n torch num threads: {torch.get_num_threads()} \
    \n ")

# ==========================
# ENV VARS
# ==========================
logger.info(f" \
    \n === IMPORTANT ENV VARS === \
    \n OMP_NUM_THREADS        = {os.environ.get("OMP_NUM_THREADS")} \
    \n MKL_NUM_THREADS        = {os.environ.get("MKL_NUM_THREADS")} \
    \n NUMEXPR_NUM_THREADS    = {os.environ.get("NUMEXPR_NUM_THREADS")} \
    \n OPENBLAS_NUM_THREADS   = {os.environ.get("OPENBLAS_NUM_THREADS")} \
    \n VECLIB_MAXIMUM_THREADS = {os.environ.get("VECLIB_MAXIMUM_THREADS")} \
    \n CUDA_VISIBLE_DEVICES   = {os.environ.get("CUDA_VISIBLE_DEVICES")} \
    \n ULTRALYTICS_CONFIG_DIR = {os.environ.get("ULTRALYTICS_CONFIG_DIR")} \
    \n ")

# ==========================
# PATH CHECKS
# ==========================
logger.info("=== PATH CHECKS ===")
print_path_status("DATA_YAML", DATA_YAML)
print_path_status("DATASET_ROOT", DATASET_ROOT)
print_path_status("PROJECT_DIR", PROJECT_DIR)

for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / split
    img_dir = split_dir / "images"
    lbl_dir = split_dir / "labels"

    print_path_status(f"{split.upper()} DIR", split_dir)
    logger.info(f"{split.upper()} COUNTS:")
    logger.info("  image files =", count_files(img_dir, {".jpg", ".jpeg", ".png", ".bmp", ".webp"}))
    logger.info("  label files =", count_files(lbl_dir, {".txt"}))
    logger.info()

# ==========================
# YAML CHECK
# ==========================
logger.info("=== YAML CONTENTS ===")
with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

logger.info(json.dumps(data_cfg, indent=2))

logger.info()
logger.info("=== YAML PATH VALIDATION ===")
for key in ["train", "val", "test"]:
    p = Path(data_cfg[key])
    logger.info(f"{key}: {p} | exists={p.exists()} | is_dir={p.is_dir()}")

In [ ]:
## Verify YOLO26 is accessible and usable

try:
    YOLO("yolo26n.yaml")
    logger.info("DONE! Model config recognized")
except Exception as e:
    logger.info("ERROR! Model config NOT recognized")
    logger.info(e)

In [ ]:
## Load parameters from specified file location

file_location =  str(PROJECT_ROOT / "config/YOLO26_config.json")
cfg = config_IO.load_json_config(file_location)

MODEL_CFG = cfg["model_cfg"]
DATA_YAML = cfg["data_yaml"]
PROJECT_DIR = cfg["project_dir"]
RUN_NAME = cfg["run_name"]
TRAIN_PARAMS = cfg["train_params"]

logger.info("MODEL_CFG =", MODEL_CFG)
logger.info("DATA_YAML =", DATA_YAML)
logger.info("PROJECT_DIR =", PROJECT_DIR)
logger.info("RUN_NAME =", RUN_NAME)
logger.info("TRAIN_PARAMS =", TRAIN_PARAMS)

In [ ]:
## Manual overrides

## for example:
# cfg["run_name"] = "tomato_flower_yolo26_trial_02"
# cfg["train_params"]["epochs"] = 150

## We change the configs within the 'cfg' object as opposed to directly on MODEL_CFG or other imported vars.
## This is so, whenever we save the configs, they can be exported back to the JSON config file if we wish.

In [ ]:
## Training the YOLO26 model

model = YOLO(cfg["model_cfg"])

results = model.train(
    data=cfg["data_yaml"],
    project=cfg["project_dir"],
    name=cfg["run_name"],
    **cfg["train_params"]
)

logger.info("DONE! Training complete!")

In [ ]:
## Save current config values backk to specified config json file.

CONFIG_PATH = str(PROJECT_ROOT / "config/YOLO26_config.json")
config_IO.save_json_config(cfg, CONFIG_PATH)